In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("¡Todas las librerías se cargaron correctamente!")

¡Todas las librerías se cargaron correctamente!


In [ ]:
df = pd.read_csv('mexico_covid19.csv')
#df.tail()

# --- Useful Pandas commands to inspect the dataset (Comandos para inspeccionar) ---
# df.head()          # Display first 5 rows / Ver primeras 5 filas
#df.info()          # Display column types, non-null counts & memory usage / Ver tipos de datos y nulos
# df.describe()      # Summary statistics (mean, min, max, std) / Estadísticas descriptivas
# df.columns         # List all 41 column names / Listar nombres de las columnas
# df.shape           # Display dimensions (rows, columns) / Ver número de filas y columnas
# df.isnull().sum()  # Count missing (NaN) values per column / Contar valores nulos por columna
# df['TIPO_PACIENTE'].value_counts() # Check target distribution / Ver distribución de hospitalizados vs ambulatorios
#df.sample(10)
#df.shape
#df.info()
#df.head(10)
#df.describe()
#df.describe(include='all')
#df.isnull().sum() 
#df['OTRO_CASO'].value_counts()
#df.nunique()
#df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263007 entries, 0 to 263006
Data columns (total 41 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   263007 non-null  int64  
 1   FECHA_ARCHIVO        263007 non-null  object 
 2   ID_REGISTRO          263007 non-null  object 
 3   ENTIDAD_UM           263007 non-null  int64  
 4   ENTIDAD_RES          263007 non-null  int64  
 5   RESULTADO            263007 non-null  int64  
 6   DELAY                263007 non-null  int64  
 7   ENTIDAD_REGISTRO     263007 non-null  int64  
 8   ENTIDAD              263007 non-null  object 
 9   ABR_ENT              263007 non-null  object 
 10  FECHA_ACTUALIZACION  263007 non-null  object 
 11  ORIGEN               263007 non-null  int64  
 12  SECTOR               263007 non-null  int64  
 13  SEXO                 263007 non-null  int64  
 14  ENTIDAD_NAC          263007 non-null  int64  
 15  MUNICIPIO_RES    

# --- FASE 1: AUDITORÍA DE FUGA DE DATOS Y CLASIFICACIÓN DE COLUMNAS ---

En este primer paso metodológico, dividimos formalmente las **41 columnas del dataset** en tres categorías clave para garantizar que nuestro modelo aprenda únicamente con información real y honesta, evitando el grave error metodológico de la **Fuga de Datos (Data Leakage)**.

### 1. 🎯 Variable Objetivo (Target)
* **`TIPO_PACIENTE`**: Es lo que queremos predecir en el *Minuto 1* del Triage clínico.
  * `1` = Paciente Ambulatorio (Leve)
  * `2` = Paciente Hospitalizado (Grave)

### 2. 🚫 Variables Excluidas por Fuga de Datos (Leakage) e Irrelevancia
* **Fuga de Datos (Eventos futuros o confirmatorios tardíos):** `UCI`, `INTUBED`, `PNEUMONIA`, `FECHA_DEF`, `RESULTADO`.
  * **¿Por qué se excluyen rotundamente?** Un modelo de triage se utiliza en el *Minuto 1*, justo cuando el paciente llega a urgencias. En ese momento **no sabemos** si el paciente acabará en la UCI días después, si necesitará intubación, si morirá o cuál será el resultado final del test PCR de laboratorio (que suele tardar días). Si incluyéramos estas variables, el modelo haría "trampa" (*fuga de información del futuro*) y daría una precisión engañosa en pruebas, pero **sería completamente inútil en un hospital real**.
* **Variables Administrativas / Identificadores:** `id`, `FECHA_ARCHIVO`, `ID_REGISTRO`, `ENTIDAD_UM`, `ENTIDAD_RES`, `ENTIDAD_REGISTRO`, `ENTIDAD`, `ABR_ENT`, `MUNICIPIO_RES`, `LOCALIDAD_RES`, `FECHA_ACTUALIZACION`, `ORIGEN`, `SECTOR`, `ENTIDAD_NAC`, `DELAY`, `FECHA_INGRESO`, `FECHA_SINTOMAS`.
  * **¿Por qué se excluyen?** Son códigos administrativos o fechas de reporte que no reflejan el estado biológico ni la gravedad clínica del paciente al llegar al hospital.

### 3. ✅ Variables Predictoras Válidas (Features del Minuto 1)
* Son todas aquellas variables clínicas, síntomas y datos demográficos disponibles de forma inmediata al evaluar al paciente en urgencias: **`EDAD`, `SEXO`, `EMBARAZO`, `DIABETES`, `EPOC`, `ASMA`, `INMSUPR`, `HIPERTENSION`, `CARDIOVASCULAR`, `OBESIDAD`, `RENAL_CRONICA`, `TABAQUISMO`, `OTRA_COM`, `OTRO_CASO`, `MIGRANTE`, `PAIS_NACIONALIDAD`, `PAIS_ORIGEN`, `INDIGENA`, `HABLA_LENGUA_INDIG`**.

> 💡 **Nota para ti:** Si en algún momento quieres modificar esta clasificación (por ejemplo, para experimentar qué pasa si incluyes una columna administrativa), puedes editar manualmente las listas en la celda de código de abajo.

In [15]:
# --- CLASIFICACIÓN Y TRADUCCIÓN DE LAS 41 COLUMNAS ---
# Definimos las listas en Python. ¡Recuerda que puedes modificarlas manualmente cuando quieras!

target = 'TIPO_PACIENTE'

# 1. Variables Excluidas por Fuga de Datos (Data Leakage - Eventos futuros o confirmación tardía)
variables_fuga = [
    'UCI',          # Ingreso a Cuidados Intensivos (complicación hospitalaria posterior)
    'INTUBADO',      # Paciente intubado (procedimiento posterior al triage)
    'FECHA_DEF',    # Fecha de defunción (evento futuro incompatible con predicción temprana)
    'RESULTADO'     # Resultado confirmatorio de laboratorio (no disponible al minuto 1 en urgencias)
]

# 2. Variables Administrativas, Fechas de reporte y Códigos de ubicación (sin señal clínica del paciente)
variables_administrativas = [
    'id', 'FECHA_ARCHIVO', 'ID_REGISTRO', 'ENTIDAD_UM', 'ENTIDAD_RES', 
    'ENTIDAD_REGISTRO', 'ENTIDAD', 'ABR_ENT', 'MUNICIPIO_RES', 
    'FECHA_ACTUALIZACION', 'ORIGEN', 'SECTOR', 'ENTIDAD_NAC', 'DELAY', 
    'FECHA_INGRESO', 'FECHA_SINTOMAS', 'NACIONALIDAD', 'EMBARAZO', 'HABLA_LENGUA_INDIG','MIGRANTE','PAIS_NACIONALIDAD','PAIS_ORIGEN'
]

# 3. Variables Predictoras Válidas (Disponibles en el Minuto 1 de Triage clínico)
# Calculamos automáticamente las predictoras tomando todas las que no estén en las listas anteriores:
columnas_excluidas = [target] + variables_fuga + variables_administrativas
predictoras_validas = [col for col in df.columns if col not in columnas_excluidas]

# --- VERIFICACIÓN Y AUDITORÍA METODOLÓGICA ---
print("=== RESUMEN DE AUDITORÍA DE COLUMNAS ===")
print(f"🎯 Variable Objetivo (Target): 1 ({target})")
print(f"🚫 Variables excluidas por Fuga de Datos (Leakage): {len(variables_fuga)} -> {variables_fuga}")
print(f"📂 Variables Administrativas / Identificadores excluidos: {len(variables_administrativas)}")
print(f"✅ Variables Predictoras Válidas (Minuto 1): {len(predictoras_validas)}")

print("\n=== LISTADO FINAL DE PREDICTORAS VÁLIDAS ===")
for i, col in enumerate(predictoras_validas, 1):
    print(f"  {i:2d}. {col}")

# Verificamos la suma total de columnas:
total_clasificadas = 1 + len(variables_fuga) + len(variables_administrativas) + len(predictoras_validas)
print(f"\nTotal columnas en el dataset: {df.shape[1]} | Total columnas clasificadas: {total_clasificadas}")

if df.shape[1] == total_clasificadas:
    print("✨ ¡AUDITORÍA EXITOSA! El 100% de las columnas han sido clasificadas sin duplicados ni omisiones.")
else:
    print("⚠️ ATENCIÓN: Revisa las listas, hay columnas duplicadas u omitidas.")

=== RESUMEN DE AUDITORÍA DE COLUMNAS ===
🎯 Variable Objetivo (Target): 1 (TIPO_PACIENTE)
🚫 Variables excluidas por Fuga de Datos (Leakage): 4 -> ['UCI', 'INTUBADO', 'FECHA_DEF', 'RESULTADO']
📂 Variables Administrativas / Identificadores excluidos: 22
✅ Variables Predictoras Válidas (Minuto 1): 14

=== LISTADO FINAL DE PREDICTORAS VÁLIDAS ===
   1. SEXO
   2. NEUMONIA
   3. EDAD
   4. DIABETES
   5. EPOC
   6. ASMA
   7. INMUSUPR
   8. HIPERTENSION
   9. OTRA_COM
  10. CARDIOVASCULAR
  11. OBESIDAD
  12. RENAL_CRONICA
  13. TABAQUISMO
  14. OTRO_CASO

Total columnas en el dataset: 41 | Total columnas clasificadas: 41
✨ ¡AUDITORÍA EXITOSA! El 100% de las columnas han sido clasificadas sin duplicados ni omisiones.


In [16]:
# --- FASE 1: LIMPIEZA Y PREPROCESAMIENTO DE DATOS ---
# 1. Crear un sub-dataframe únicamente con el Target y las Predictoras Válidas (Cortafuegos de seguridad)
columnas_modelo = [target] + predictoras_validas
df_clean = df[columnas_modelo].copy()

print("=== 1. DIAGNÓSTICO DE CÓDIGOS ESPECIALES (97, 98, 99) ANTES DE LIMPIAR ===")
for col in df_clean.columns:
    if col != 'EDAD': # En la edad, 97, 98 y 99 son años reales, no faltantes
        conteos_especiales = df_clean[col].isin([97, 98, 99]).sum()
        if conteos_especiales > 0:
            porcentaje = (conteos_especiales / len(df_clean)) * 100
            print(f"  * {col:<15}: {conteos_especiales:8,d} registros faltantes ({porcentaje:.2f}%)")

# --- 2. BINARIZACIÓN DEL TARGET (TIPO_PACIENTE) ---
# Original: 1 = Ambulatorio, 2 = Hospitalizado
# Mapeo ML: 0 = Ambulatorio (clase referencia), 1 = Hospitalizado (clase de interés / riesgo)
df_clean[target] = df_clean[target].replace({1: 0, 2: 1})

# --- 3. RE-CODIFICACIÓN DE BOOLEANOS Y TRATAMIENTO DE NULOS ---
# Identificamos todas las columnas booleanas (todas excepto EDAD)
cols_booleanas = [col for col in predictoras_validas if col != 'EDAD']

for col in cols_booleanas:
    # Paso A: Convertir códigos de faltantes (97: No aplica, 98: Se ignora, 99: No especificado) a NaN de numpy
    df_clean[col] = df_clean[col].replace([97, 98, 99], np.nan)
    
    # Paso B: Convertir '2 = No' en '0' (y en SEXO: 2 = Hombre pasa a 0, 1 = Mujer se queda como 1)
    # Los valores 1 (Sí) se quedan exactamente como 1.
    df_clean[col] = df_clean[col].replace(2, 0)

# --- 4. VERIFICACIÓN DE EDAD ---
# Asegurarnos de que no haya edades negativas o absurdas (> 120 años)
edades_invalidas = df_clean[df_clean['EDAD'] < 0].shape[0] + df_clean[df_clean['EDAD'] > 120].shape[0]
if edades_invalidas > 0:
    print(f"\n⚠️ Se encontraron {edades_invalidas} registros con edades inválidas (<0 o >120). Se convertirán en NaN.")
    df_clean.loc[(df_clean['EDAD'] < 0) | (df_clean['EDAD'] > 120), 'EDAD'] = np.nan

# --- VERIFICACIÓN FINAL DEL DATASET LIMPIO ---
print("\n=== 2. AUDITORÍA DEL DATASET LIMPIO (df_clean) ===")
print(f"Dimensiones finales: {df_clean.shape[0]:,d} filas x {df_clean.shape[1]} columnas")
print("\nDistribución del Target Binarizado (TIPO_PACIENTE):")
conteo_target = df_clean[target].value_counts(dropna=False)
for val, count in conteo_target.items():
    etiqueta = "Hospitalizado (1)" if val == 1 else "Ambulatorio (0)"
    print(f"  * {etiqueta:<18}: {count:10,d} ({count/len(df_clean)*100:.2f}%)")

print("\nConteo total de valores nulos (NaN) resultantes por columna:")
nulos_por_col = df_clean.isnull().sum()
print(nulos_por_col[nulos_por_col > 0])
print("\n✨ ¡Limpieza completada con éxito! Todas las variables están estandarizadas (0 y 1) y listas para ML.")


=== 1. DIAGNÓSTICO DE CÓDIGOS ESPECIALES (97, 98, 99) ANTES DE LIMPIAR ===
  * NEUMONIA       :       14 registros faltantes (0.01%)
  * DIABETES       :    1,011 registros faltantes (0.38%)
  * EPOC           :      931 registros faltantes (0.35%)
  * ASMA           :      923 registros faltantes (0.35%)
  * INMUSUPR       :    1,038 registros faltantes (0.39%)
  * HIPERTENSION   :      938 registros faltantes (0.36%)
  * OTRA_COM       :    1,344 registros faltantes (0.51%)
  * CARDIOVASCULAR :      961 registros faltantes (0.37%)
  * OBESIDAD       :      962 registros faltantes (0.37%)
  * RENAL_CRONICA  :      937 registros faltantes (0.36%)
  * TABAQUISMO     :      982 registros faltantes (0.37%)
  * OTRO_CASO      :   83,114 registros faltantes (31.60%)

=== 2. AUDITORÍA DEL DATASET LIMPIO (df_clean) ===
Dimensiones finales: 263,007 filas x 15 columnas

Distribución del Target Binarizado (TIPO_PACIENTE):
  * Ambulatorio (0)   :    200,838 (76.36%)
  * Hospitalizado (1) :     62